# Chronos-2 Promotion Forecast Lab (Synthetic)

## Source Attribution

- **Upstream notebook**: [amazon-science/chronos-forecasting — chronos-2-quickstart.ipynb](https://github.com/amazon-science/chronos-forecasting/blob/main/notebooks/chronos-2-quickstart.ipynb)
- **Model**: `autogluon/chronos-2-small` at revision `ddec01313e50b6bc58ebaa92ede81bc24a3d9f9a` (Apache-2.0)
- **Package**: `chronos-forecasting` (Apache-2.0)

## Synthetic Adaptation

This notebook is an **explicitly authorized adaptation** of the upstream covariate-forecasting method to **synthetic data**. No retail (Rossmann) or electricity datasets are downloaded, redistributed, or used. All series are generated locally from a seeded RNG.

The synthetic fixture models a sales signal with:
- A linear trend
- Weekly seasonality (sin)
- 28-day cycle seasonality
- Promotion uplift (40 units during promotion windows)
- Gaussian noise (σ=2.5)

The held-out actual is generated with the user-selected promotion schedule. The seasonal-7 baseline repeats the last 7 historical observations.

> **Note**: This is a scenario forecast, not a causal estimate or a production-accuracy claim. The p10–p90 interval is not calibrated on this fixture.

## Stages

1. **Prepare** synthetic history and known future covariates
2. **Infer** with real Chronos-2 Small (CPU, float32) and compute seasonal-7 baseline
3. **Evaluate** MAE, baseline MAE, and p10–p90 coverage; persist `results.json`


In [ ]:
import os
import sys
import json

# Honor supplied PROJECT_ROOT first, otherwise env, then notebook cwd
if "PROJECT_ROOT" not in globals() or globals()["PROJECT_ROOT"] is None:
    PROJECT_ROOT = os.environ.get("PROJECT_ROOT", os.getcwd())

# Normalize to a filesystem string (sys.path requires str, not pathlib.Path)
PROJECT_ROOT = os.fspath(PROJECT_ROOT)

# Bootstrap path for forecast_core import (temporary, restored)
_saved = sys.path[:]
try:
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    import forecast_core
finally:
    sys.path[:] = _saved

import numpy as np
import pandas as pd

make_fixture = forecast_core.make_fixture
predict = forecast_core.predict
run_analysis = forecast_core.run_analysis
MODEL_ID = forecast_core.MODEL_ID
MODEL_REVISION = forecast_core.MODEL_REVISION


In [ ]:
# Exactly one analysis/inference call (run_analysis internally invokes predict once)
seed = 42
horizon = 28
promotion_start = 7
promotion_days = 7

result = run_analysis(seed=seed, horizon=horizon, promotion_start=promotion_start, promotion_days=promotion_days)

# Validate: all metric values must be finite
for key in ["mae", "baseline_mae", "coverage"]:
    val = result["metrics"][key]
    assert np.isfinite(val), f"Non-finite value for {key}: {val}"

# Write results.json to current execution directory
output = {
    "results": {
        "mae": result["metrics"]["mae"],
        "baseline_mae": result["metrics"]["baseline_mae"],
        "coverage": result["metrics"]["coverage"],
        "horizon": result["horizon"],
        "seed": result["seed"],
    }
}

with open("results.json", "w") as f:
    json.dump(output, f, indent=2)

print("Wrote results.json")
print(json.dumps(output, indent=2))
